# 02 · ¿Qué hay detrás de los datos raros?

> **Reto 2 del INE:** "¿Qué zonas podrían presentar mayor abstención en el futuro si no se interviene?"

En el paso 1 encontramos datos raros: municipios donde "no sabemos" lo que hizo casi nadie, años con partes de la lista faltantes, participación casi nula o sospechosamente alta, personas de 140 años. **Aquí investigamos por qué**, con tres tipos de evidencia:

1. **Los propios datos.**
2. **Las actas oficiales de 2024:** los resultados por casilla del INE (cómputos distritales), una segunda fuente independiente.
3. **Noticias, sentencias del Tribunal Electoral y documentos del INE.**

Cada hallazgo indica su fuente y si **lo verificamos nosotros** o viene de una fuente que solo citamos. Los reportes completos, con todas las fuentes, están en `docs/investigacion/` (A a F).

## Resumen de lo que encontramos

1. **El "no sabemos" tiene tres orígenes distintos**, y uno de ellos **es abstención real**: municipios donde no se instalaron casillas, por decisión de la comunidad o por violencia. En Cherán, solo el 7.1 % de la lista votó en 2024, aunque la fórmula dé 47.1 %. **Esto cambia la decisión D3.**
2. **El hueco de 2012 lo explica el INE:** 5,485 cuadernillos no se pudieron capturar y quedaron fuera de la base.
3. **El "no sabemos" de 2015 coincide con el boicot magisterial en Oaxaca y Guerrero.**
4. **Hay secciones donde nuestros datos marcan más "votó" que los votos del acta oficial.** No deben entrar al modelo.
5. **Las personas de 110 años o más ya las revisó el INE:** son sobre todo edades exageradas en actas de nacimiento antiguas.
6. **Los movimientos sospechosos de electores son muy raros:** casi todo el crecimiento atípico de la lista es urbano. Solo un caso combina crecimiento atípico con salto de participación.
7. **Elegir gobernador el mismo día cambia mucho la participación de la intermedia:** entre 2012 y 2015 cayó 4.3 pp en los estados que lo eligieron y 17.6 pp en los demás. En 2027 se eligen 17 gubernaturas.

In [1]:
from pathlib import Path
import io
import zipfile
import hashlib
import urllib.request

import polars as pl

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(12)
pl.Config.set_fmt_str_lengths(40)
pl.Config.set_tbl_width_chars(170)

raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file() and (p / 'data').is_dir())
datos = pl.scan_parquet(raiz / 'data/processed/ccpc/ccpc-*.parquet')

### Actas oficiales de 2024

Son los resultados de la elección presidencial **casilla por casilla**, publicados por el INE: lista nominal de la casilla, votos y estado (instalada, no instalada, paquete no recibido).

Si no están en la computadora, la siguiente celda las descarga del sitio del INE (18 MB) a `data/raw/computos_2024/`, carpeta que no se sube al repositorio.

In [2]:
carpeta = raiz / 'data/raw/computos_2024'
archivo = carpeta / 'PRES_2024.csv'
if not archivo.exists():
    carpeta.mkdir(parents=True, exist_ok=True)
    peticion = urllib.request.Request('https://computos2024.ine.mx/20240608_2030_COMPUTOS.zip', headers={'User-Agent': 'Mozilla/5.0'})
    contenido = urllib.request.urlopen(peticion, timeout=120).read()
    (carpeta / 'COMPUTOS_2024.zip').write_bytes(contenido)
    with zipfile.ZipFile(io.BytesIO(contenido)) as z:
        interno = next(n for n in z.namelist() if n.endswith('PRES.zip'))
        with zipfile.ZipFile(io.BytesIO(z.read(interno))) as zp:
            archivo.write_bytes(zp.read('PRES_2024.csv'))
print('SHA-256:', hashlib.sha256(archivo.read_bytes()).hexdigest())

lineas = archivo.read_text(encoding='latin-1').split('\n')
inicio = next(i for i, linea in enumerate(lineas) if linea.startswith('CLAVE_CASILLA'))
actas = (
    pl.read_csv(io.StringIO('\n'.join(lineas[inicio:])), separator='|', infer_schema=False, truncate_ragged_lines=True)
    .with_columns(
        EDOCVE=pl.col('ID_ENTIDAD').str.replace_all(r'[="]', '').cast(pl.Int8, strict=False),
        SECCION=pl.col('SECCION').str.replace_all(r'[="]', '').cast(pl.Int32, strict=False),
        votos=pl.col('TOTAL_VOTOS_CALCULADOS').cast(pl.Int64, strict=False),
        lista_casilla=pl.col('LISTA_NOMINAL').cast(pl.Int64, strict=False),
        estado=pl.when(pl.col('OBSERVACIONES').str.starts_with('Casilla no instalada')).then(pl.lit('no instalada'))
        .when(pl.col('OBSERVACIONES').str.starts_with('Paquete no recibido')).then(pl.lit('paquete no recibido'))
        .otherwise(pl.lit('computada')),
    )
    # Básicas, contiguas y extraordinarias: las que corresponden a la lista nominal de la sección
    .filter(pl.col('TIPO_CASILLA').is_in(['B', 'C', 'E']))
)
print(f'Casillas en las actas: {actas.height:,}')
print(actas.group_by('estado').len().sort('len', descending=True))

SHA-256: 860c1a539ac98b5def4dff3e82e87a19be3ebba7cb435b680ae79799942d1628
Casillas en las actas: 169,003
shape: (3, 2)
┌─────────────────────┬────────┐
│ estado              ┆ len    │
│ ---                 ┆ ---    │
│ str                 ┆ u32    │
╞═════════════════════╪════════╡
│ computada           ┆ 168859 │
│ paquete no recibido ┆ 121    │
│ no instalada        ┆ 23     │
└─────────────────────┴────────┘


## 1 · El "no sabemos" tiene tres orígenes

En el paso 1 tratamos el "no sabemos" como información perdida. Al cruzarlo con las actas aparecen **tres situaciones distintas**:

| Origen | Qué pasó | ¿Qué significa para el reto? |
|---|---|---|
| **A. No hubo casilla** | La casilla no se instaló o se dio de baja antes de la elección (decisión de la comunidad, conflicto, violencia) | **Es abstención real:** esas personas no votaron. Señal fuerte de riesgo |
| **B. Se votó, pero el paquete no llegó** | Quema, robo o paquete perdido. El cuadernillo viaja dentro del paquete | Señal de conflicto. Las actas, si existen, dicen cuántos votaron |
| **C. Se votó normal, pero el cuadernillo no se capturó** | Las actas están completas y la participación es normal | Dato perdido por causas administrativas. Las actas dan la participación real |

**Cómo se distinguen con las actas de 2024**, para cada municipio:

- **Lista sin casilla:** la parte de su lista que no quedó cubierta por ninguna casilla que funcionó. Por ejemplo, si una sección tiene 1,000 personas y solo se instaló la casilla de 500, las otras 500 se quedaron sin casilla. Origen A.
- **Lista con paquete no recibido:** origen B.
- **Participación según las actas:** votos del acta / lista nominal **completa** del municipio.

In [3]:
seccion_2024 = datos.filter(pl.col('AELEC') == 2024).group_by('EDOCVE', 'MPIOCVE', 'SECCION').agg(
    pl.col('MPIONOM').first().alias('municipio'), pl.col('EDONOM').first().alias('estado_nombre'), pl.col('LN', 'SV', 'NV', 'NS').sum()
).collect()
actas_seccion = actas.group_by('EDOCVE', 'SECCION').agg(
    lista_computada=pl.col('lista_casilla').filter(pl.col('estado') == 'computada').sum(),
    lista_no_instalada=pl.col('lista_casilla').filter(pl.col('estado') == 'no instalada').sum(),
    lista_paquete_perdido=pl.col('lista_casilla').filter(pl.col('estado') == 'paquete no recibido').sum(),
    votos=pl.col('votos').filter(pl.col('estado') == 'computada').sum(),
    casillas=pl.len(),
)
cruce = seccion_2024.join(actas_seccion, on=['EDOCVE', 'SECCION'], how='left').with_columns(
    pl.col('lista_computada', 'lista_no_instalada', 'lista_paquete_perdido', 'votos', 'casillas').fill_null(0)
).with_columns(
    # Lista sin casilla: la parte de la lista de la sección que no quedó cubierta por ninguna casilla
    # que funcionó (computada o con paquete perdido). Incluye casillas no instaladas y dadas de baja antes de la elección.
    sin_casilla=(pl.col('LN') - pl.col('lista_computada') - pl.col('lista_paquete_perdido')).clip(lower_bound=0)
)
por_municipio = cruce.group_by('EDOCVE', 'MPIOCVE').agg(
    pl.col('municipio').first(), pl.col('estado_nombre').first(),
    pl.col('LN', 'SV', 'NV', 'NS', 'votos', 'sin_casilla', 'lista_paquete_perdido').sum(),
).with_columns(
    no_sabemos=(pl.col('NS') / pl.col('LN') * 100).round(1),
    p_formula=pl.when(pl.col('SV') + pl.col('NV') > 0).then(pl.col('SV') / (pl.col('SV') + pl.col('NV')) * 100).round(1),
    p_actas=(pl.col('votos') / pl.col('LN') * 100).round(1),
    sin_casilla_pct=(pl.col('sin_casilla') / pl.col('LN') * 100).round(1),
    paquete_pct=(pl.col('lista_paquete_perdido') / pl.col('LN') * 100).round(1),
).with_columns(
    origen=pl.when(pl.col('sin_casilla_pct') >= 25).then(pl.lit('A · no hubo casilla'))
    .when(pl.col('paquete_pct') >= 25).then(pl.lit('B · paquete no recibido'))
    .otherwise(pl.lit('C · no capturado'))
)
alto_ns = por_municipio.filter(pl.col('NS') / pl.col('LN') > 0.5)
print(f'Municipios con más de la mitad en "no sabemos" en 2024: {alto_ns.height}')
print(alto_ns.group_by('origen').len().sort('origen'))
alto_ns.sort('origen', 'p_actas').select(
    pl.col('estado_nombre').alias('Estado'), pl.col('municipio').alias('Municipio'), pl.col('LN').alias('Lista'),
    pl.col('no_sabemos').alias('No sabemos (%)'), pl.col('sin_casilla_pct').alias('Lista sin casilla (%)'),
    pl.col('p_formula').alias('Participación según la fórmula (%)'), pl.col('p_actas').alias('Votos del acta / toda la lista (%)'),
    pl.col('origen').alias('Origen'),
)

Municipios con más de la mitad en "no sabemos" en 2024: 38
shape: (3, 2)
┌─────────────────────────┬─────┐
│ origen                  ┆ len │
│ ---                     ┆ --- │
│ str                     ┆ u32 │
╞═════════════════════════╪═════╡
│ A · no hubo casilla     ┆ 10  │
│ B · paquete no recibido ┆ 5   │
│ C · no capturado        ┆ 23  │
└─────────────────────────┴─────┘


Estado,Municipio,Lista,No sabemos (%),Lista sin casilla (%),Participación según la fórmula (%),Votos del acta / toda la lista (%),Origen
str,str,i64,f64,f64,f64,f64,str
"""CHIAPAS""","""Chicomuselo""",26330,100.0,100.0,null,0.0,"""A · no hubo casilla"""
"""OAXACA""","""Capulálpam de Méndez""",1215,100.0,100.0,null,0.0,"""A · no hubo casilla"""
"""CHIAPAS""","""Pantelhó""",15256,100.0,100.0,null,0.0,"""A · no hubo casilla"""
"""OAXACA""","""La Reforma""",2523,100.0,100.0,null,0.0,"""A · no hubo casilla"""
"""MICHOACAN""","""Cherán""",15001,88.9,84.9,47.1,7.1,"""A · no hubo casilla"""
"""OAXACA""","""San Mateo del Mar""",10676,64.6,37.0,20.3,13.9,"""A · no hubo casilla"""
"""OAXACA""","""Santa María Chimalapa""",5465,53.9,53.9,53.1,24.5,"""A · no hubo casilla"""
"""MICHOACAN""","""Charapan""",10243,65.5,44.4,75.9,41.6,"""A · no hubo casilla"""
"""MICHOACAN""","""Ziracuaretiro""",14251,100.0,27.3,null,44.4,"""A · no hubo casilla"""


**Qué vemos:** compara las dos últimas columnas numéricas.

- **Origen A (no hubo casilla):** la fórmula da una participación mucho mayor que la real. En **Cherán**, la fórmula dice 47.1 %, pero solo votó el 7.1 % de su lista, porque solo hubo casillas para una parte. En **Chicomuselo y Pantelhó** no hubo ninguna casilla.
- **Origen C (no capturado):** las actas muestran una participación normal. El "no sabemos" fue un problema de captura, no de la gente.

**Qué significa:** con la regla del paso 1 ("más de la mitad sin dato → sin dato"), **los municipios de origen A habrían quedado en gris**, cuando son justo los de mayor abstención real.

### Lo que dicen las fuentes sobre cada caso

| Municipio | Elección | Qué pasó | Fuente | ¿Verificado por nosotros? |
|---|---|---|---|---|
| Cherán (Mich.) | 2015, 2018, 2021, 2024 | El Concejo Mayor (autogobierno) no permite casillas. En 2018 no se instalaron 18 | Quadratín 6-may-2015; Excélsior 1-jul-2018; Animal Político 2-jun-2024 | 2024 sí, con las actas |
| Cherán (Mich.) | 2012 | No hubo casillas; 1,688 personas votaron en casillas especiales | Proceso, 24-abr-2018 | No |
| San Mateo del Mar (Oax.) | 2012 | La asamblea del pueblo ikoots no permitió casillas; el consejo distrital eliminó 14 | Tribunal Electoral, SUP-RRV-11/2012 | Sí, en el texto de la sentencia |
| San Mateo del Mar (Oax.) | 2024 | La asamblea no permitió 6 casillas | Excélsior, 30-may-2024 | Las actas sí (13.9 % real) |
| Chicomuselo y Pantelhó (Chis.) | 2024 | Se dieron de baja 45 y 28 casillas por violencia | Acuerdo INE/CG844/2024 | Las actas sí: sin casillas |
| Nahuatzen (Mich.) | 2018, 2024 | 12 casillas dadas de baja en 2018 (violencia y concejo indígena); negativa comunitaria en 2024 | Tribunal Electoral, SUP-REC-1061/2018 | 2024 sí (49.0 % real) |
| Bolaños (Jal.) | 2018, 2021 | El pueblo wixárika acordó no votar por un conflicto agrario | Tribunal Electoral, SUP-REC-1401/2018; Zeta 6-jun-2021 | No |
| Oxchuc, Venustiano Carranza, Honduras de la Sierra, Siltepec (Chis.) | 2021 | Casillas no instaladas por gobierno comunitario, conflicto agrario o grupos armados | El Universal 3 y 6-jun-2021; Expansión 4-jun-2021 | No |
| Rincón Chamula San Pedro (Chis.) | 2018, 2024 | Tiroteo y quema de casillas (2018); ataque armado durante el conteo (2024) | Tribunal Electoral, SX-JDC-848/2018; Milenio 3-jun-2024 | No |
| Istmo de Oaxaca, Yucatán, Puebla | 2021 | Quema de paquetes después de votar (Xadani, Reforma de Pineda, Yaxkukul, Uayma, Huehuetlán, Chiconcuautla) | Prensa local y acuerdos IEPAC e INE/CG1547/2021 | No |

**Lo que no encontramos:** la causa del "no sabemos" en distritos completos con votación normal (Coahuila distrito 07 y Oaxaca distrito 05 en 2024; Hidalgo distrito 04 y Veracruz distrito 17 en 2018). Todo indica que es administrativo (origen C).

Las fuentes completas están en los reportes de investigación, en `docs/investigacion/`.

## 2 · Por qué faltan partes de la lista en 2012

En el paso 1 vimos que en 2012 faltan 1,048 secciones completas y que 109 municipios tienen una lista mucho menor de lo normal, sin que aparezca como "no sabemos".

**Lo explica el propio INE.** Su *Estudio censal de la participación ciudadana en las elecciones federales de 2012* dice que la base:

> "no consideró 1,207 casillas correspondientes al voto de los mexicanos en el extranjero y casillas especiales; así como tampoco pudo contabilizar un total de 5,485 listas nominales que no fue posible capturar".

Es decir: **en 2012 los cuadernillos que no se capturaron se quitaron de la base**. A partir de 2015 sí se registran como "no sabemos". La lista de 2012 de nuestros datos coincide exactamente con la cifra del estudio:

In [4]:
datos.filter(pl.col('AELEC') == 2012).select(
    pl.col('LN').sum().alias('Lista nominal 2012 en nuestros datos'),
    (pl.col('SV').sum() / (pl.col('SV').sum() + pl.col('NV').sum()) * 100).round(2).alias('Participación 2012 (%)'),
).collect()

Lista nominal 2012 en nuestros datos,Participación 2012 (%)
i64,f64
76490962,62.08


**Qué vemos:** 76,490,962 personas y 62.08 % de participación, las mismas cifras que publica el estudio del INE. **Verificado por nosotros en el texto del estudio.**

**Qué significa:** la regla D2 (lista menor al 80 % de sus elecciones vecinas → "sin dato") es la correcta para 2012. Además, como 2027 es intermedia, 2012 pesa poco en el modelo.

**Fuente:** INE, *Estudio censal de la participación ciudadana en las elecciones federales de 2012*. https://portalanterior.ine.mx/docs/IFE-v2/DECEYEC/DECEyEC-EstudiosInvestigaciones/InvestigacionIFE/Estudio_Censal_Participacion_Ciudadana_2012.pdf

## 3 · 2015: el boicot magisterial

En 2015 hubo un boicot de la CNTE contra la elección en Oaxaca, Guerrero, Chiapas y Michoacán. **Según la prensa del 7 de junio de 2015, el INE informó 182 casillas no instaladas y 415 con la votación suspendida.**

- **Dónde coinciden los datos con los reportes:**
  - la zona mazateca de Oaxaca, con 11 municipios sin ningún dato;
  - el Istmo, donde Juchitán tiene 78 % de "no sabemos";
  - Tixtla, en Guerrero, con 64 % y quema de casillas reportada.

  El INE contó 5,118,975 registros "no censados" ese año, igual a nuestro "no sabemos".
- **Pero el boicot no lo explica todo:** solo el 25 % del "no sabemos" de 2015 está en esos cuatro estados. El resto es urbano: Cancún, Los Cabos, Hermosillo y Villahermosa, sin causa encontrada.

**Fuentes:**

- INE, *Diagnóstico de patrones de participación atípicos 2015* (cifra de 5,118,975, verificada por nosotros en el texto).
- La Silla Rota y Expansión, 7 de junio de 2015 (casillas no instaladas).

In [5]:
datos.filter(pl.col('AELEC') == 2015).select(pl.col('NS').sum().alias('"No sabemos" 2015 en nuestros datos')).collect()

"""No sabemos"" 2015 en nuestros datos"
i64
5118976


**Qué vemos:** 5,118,976, uno más que la cifra del INE (5,118,975).

**Qué significa:** en 2015, parte del "no sabemos" es abstención por conflicto (origen A) y parte es captura (origen C). Para separarlos hacen falta las actas de 2015.

## 4 · Participación sospechosamente alta: segunda fuente

¿Las participaciones de 100 % de Guerrero en 2024 son reales? Se compara, sección por sección, cuántas personas marcan nuestros datos como "votó" contra los votos del acta oficial. **Solo se comparan secciones donde las dos fuentes tienen la misma lista nominal y nadie quedó como "no sabemos"**, para comparar lo mismo.

In [6]:
comparables = cruce.filter((pl.col('LN') == pl.col('lista_computada')) & (pl.col('NS') == 0) & (pl.col('lista_paquete_perdido') == 0))
print(f'Secciones comparables: {comparables.height:,}')
print(f'Mediana de "votó" en nuestros datos / votos del acta: {comparables.select((pl.col("SV") / pl.col("votos")).median()).item():.3f}')
no_cuadran = comparables.filter((pl.col('SV') > pl.col('votos') * 1.10) & (pl.col('SV') - pl.col('votos') >= 50))
print(f'Secciones con más de 10 % (y al menos 50) más "votó" que votos del acta: {no_cuadran.height}')
print()
print('Secciones con 100 % de participación en Guerrero y Chiapas (2024), contra su acta:')
cruce.filter(pl.col('EDOCVE').is_in([7, 12]) & (pl.col('NV') == 0) & (pl.col('SV') >= 100)).select(
    pl.col('municipio').alias('Municipio'), pl.col('SECCION').alias('Sección'), pl.col('LN').alias('Lista'),
    pl.col('SV').alias('"Votó" en nuestros datos'), pl.col('votos').alias('Votos del acta'),
).sort('Lista', descending=True)

Secciones comparables: 61,139
Mediana de "votó" en nuestros datos / votos del acta: 0.992
Secciones con más de 10 % (y al menos 50) más "votó" que votos del acta: 359

Secciones con 100 % de participación en Guerrero y Chiapas (2024), contra su acta:


Municipio,Sección,Lista,"""Votó"" en nuestros datos",Votos del acta
str,i32,i64,i64,i64
"""Malinaltepec""",1704,1044,1044,927
"""Técpan de Galeana""",2290,540,540,541
"""San Miguel Totolapan""",2100,394,394,394
"""San Miguel Totolapan""",2105,287,287,287
"""Atoyac de Álvarez""",642,239,239,123
"""San Miguel Totolapan""",2109,141,141,142
"""Teloloapan""",2416,133,133,133


**Qué vemos:**

- **En general las dos fuentes coinciden:** la mediana de la razón es casi 1.
- **San Miguel Totolapan cuadra exacto** (394 de 394 y 287 de 287). Su participación alta es real.
  - Según el Instituto Electoral de Guerrero, ese año hubo una elección municipal muy reñida, PRI 7,585 contra PT 4,829 (fuente citada por el reporte de investigación, no verificada por nosotros). Es una explicación posible.
- **En otras secciones no cuadra.** Atoyac de Álvarez 642 marca 239 personas como "votó" y el acta tiene 123 votos; Malinaltepec 1704 marca 1,044 y el acta 927. Hay secciones así en todo el país.

**Qué significa:**

- **Una participación de 100 % no es automáticamente un error.** Hay que comprobarla con el acta.
- **Cuando nuestros datos marcan bastante más "votó" que votos el acta**, el dato de esa sección no es confiable: puede ser un marcado irregular del cuadernillo o un error de captura (hipótesis).

**Cómo comunicarlo:** "en estas secciones, dos fuentes oficiales no coinciden". Nunca "fraude".

**Contexto de Guerrero** (fuentes citadas, no verificadas por nosotros): Amapola (23 de julio de 2024) y Animal Político (10 de enero y 1 de noviembre de 2024) documentan control criminal en Tierra Caliente y Costa Grande, sobre todo en la **imposición de candidaturas**. No se encontraron denuncias ni impugnaciones sobre las secciones con 100 %.

## 5 · Los "casi cero" y las personas de 110 años

### Casi cero

En los tres municipios de Oaxaca con 0 votos en 2012, más Yutanduchi y Jicotlán, **no se encontró ninguna causa documentada**. El INE mismo reportó la sección de Yutanduchi como la de participación más baja de Oaxaca en 2012, sin explicarla.

- **Fuente:** INE, *Diagnóstico de patrones de participación atípicos 2012*. Verificado por el reporte de investigación, no por nosotros.
- **Hipótesis más probable:** cuadernillos sin marcas que se registraron como "no votó" en lugar de "no sabemos".
- **Se mantiene la decisión D2:** "sin dato".

### Personas de 110 años o más

El INE ya revisó a este grupo. Su Comité Técnico de Evaluación del Padrón Electoral (COTEPE 2023-2024) encontró 369 personas de más de 110 años en el padrón, concentradas en Guerrero, Chiapas y Veracruz, igual que en nuestros datos.

- **Qué encontró al revisar los documentos:**
  - en el 57 %, la edad coincide con el acta de nacimiento;
  - solo el 1.6 % tenía un error de captura del año (1888 en lugar de 1988).
- **Su conclusión:** no afectan la validez del padrón.
- **Lo que dice la demografía:** en México se exagera la edad en la vejez extrema, sobre todo en el sur. El censo también cuenta más centenarios de los que hay (Lara-Castañeda y García-Guerrero, *Demographic Research*, 2026).

**Fuentes:** COTEPE 2023-2024, https://repositoriodocumental.ine.mx/xmlui/bitstream/handle/123456789/172195/CGor202405-29-ip-23.pdf. Fuente citada, no verificada por nosotros.

**Qué significa:** no hay que hablar de "voto de muertos". Se mantiene la decisión D4: agruparlas en "100 y más".

## 6 · ¿Hay movimientos sospechosos de electores?

**La idea:** si mucha gente cambia su credencial a un municipio para votar ahí (lo que se conoce como "turismo electoral"), la lista de ese municipio crecería de forma rara y podría saltar su participación.

**Cómo se buscó:**

- Se comparó cuánto crece la lista de cada municipio contra su estado, solo en las secciones que existen en los dos años y con municipios de tamaño parecido.
- Se revisaron los pares de municipios vecinos donde uno crece y el otro cae.

**Qué se encontró:**

- **122 casos de crecimiento atípico** entre 2009 y 2024. **105 son expansión urbana sostenida** (periferias de Monterrey, Guadalajara, Querétaro, Valle de México y Riviera Maya), 6 son cambios de límites y **11 quedan "a revisar"**, casi todos en municipios de menos de 6 mil electores.
- **Los pares de vecinos** (uno crece y el otro cae) aparecen **menos** de lo que daría el azar: el crecimiento urbano se extiende a los vecinos.
- **Solo un caso combina crecimiento atípico con salto de participación:** Las Minas (Veracruz).

In [7]:
datos.filter((pl.col('EDOCVE') == 30) & (pl.col('MPIONOM') == 'Las Minas')).group_by('AELEC').agg(
    pl.col('LN').sum().alias('Lista'),
    (pl.col('SV').sum() / (pl.col('SV').sum() + pl.col('NV').sum()) * 100).round(1).alias('Participación (%)'),
).sort('AELEC').rename({'AELEC': 'Elección'}).collect()

Elección,Lista,Participación (%)
i16,i64,f64
2009,1754,56.0
2012,1807,63.4
2015,1925,52.8
2018,2082,64.8
2021,2521,87.6
2024,2484,52.8


**Qué vemos:** en 2021 la lista de Las Minas crece 21 % (su estado, 3 %) y la participación salta a 87.6 %. En 2024 vuelve a 52.8 %.

**Qué significa:**

- Es un patrón que vale la pena mirar, **pero no prueba nada**. Puede haber un fraccionamiento nuevo, una campaña de credencialización o una elección municipal reñida.
- **Para ponerlo en contexto:** con sus propios criterios, el INE solo confirmó como irregular el 13 % de los domicilios que marcó como sospechosos entre 2012 y 2023 (COTEPE 2023-2024; fuente citada).

**Recomendación:**

- **No usarlo como marca de "sospecha" en el mapa.**
- **Para el modelo**, el crecimiento de la lista sí sirve como variable: donde la lista crece rápido (periferias urbanas), se vota menos.

**Otro hallazgo de esta búsqueda:** en Hueyapan (Morelos), dos secciones pasan de 1,215 y 1,330 personas a solo **10 y 65 en 2021**, y vuelven a lo normal en 2024. Es un dato dudoso de 2021.

## 7 · La elección de gobernador el mismo día

**Por qué importa:** en una intermedia solo se eligen diputados federales, y suele votar menos gente. Pero si ese mismo día se elige gobernador, la elección atrae más votantes. **En 2027 se eligen 17 gubernaturas** (fuente oficial del INE; ver `docs/investigacion/G-calendario-2027.md`).

Se compara cuánto cae la participación de cada estado entre la presidencial y la intermedia siguiente, según si eligió gobernador o no. Las listas de estados con gobernador están verificadas en fuentes periodísticas y académicas (reporte G).

In [8]:
from scipy import stats
GOBERNADOR = {
    2009: ['CAMPECHE', 'COLIMA', 'NUEVO LEON', 'QUERETARO', 'SAN LUIS POTOSI', 'SONORA'],
    2015: ['BAJA CALIFORNIA SUR', 'CAMPECHE', 'COLIMA', 'GUERRERO', 'MICHOACAN', 'NUEVO LEON', 'QUERETARO', 'SAN LUIS POTOSI', 'SONORA'],
    2021: ['BAJA CALIFORNIA', 'BAJA CALIFORNIA SUR', 'CAMPECHE', 'CHIHUAHUA', 'COLIMA', 'GUERRERO', 'MICHOACAN', 'NAYARIT', 'NUEVO LEON',
           'QUERETARO', 'SAN LUIS POTOSI', 'SINALOA', 'SONORA', 'TLAXCALA', 'ZACATECAS'],
}
nombres_2024 = datos.filter(pl.col('AELEC') == 2024).group_by('EDOCVE').agg(pl.col('EDONOM').first().alias('estado')).collect()
estatal = (
    datos.group_by('AELEC', 'EDOCVE').agg(p=pl.col('SV').sum() / (pl.col('SV').sum() + pl.col('NV').sum()) * 100).collect()
    .join(nombres_2024, on='EDOCVE').pivot(on='AELEC', index='estado', values='p')
)
assert all(e in estatal['estado'].to_list() for lista in GOBERNADOR.values() for e in lista)
filas = []
for presidencial, intermedia in [(2012, 2015), (2018, 2021)]:
    t = estatal.with_columns(cambio=pl.col(str(intermedia)) - pl.col(str(presidencial)), gob=pl.col('estado').is_in(GOBERNADOR[intermedia]))
    con, sin = t.filter(pl.col('gob'))['cambio'], t.filter(~pl.col('gob'))['cambio']
    filas.append({'Periodo': f'{presidencial} → {intermedia}', 'Estados con gobernador': con.len(), 'Cambio promedio con gobernador (pp)': round(con.mean(), 1),
                  'Estados sin gobernador': sin.len(), 'Cambio promedio sin gobernador (pp)': round(sin.mean(), 1),
                  'Valor p (prueba de Mann-Whitney)': round(stats.mannwhitneyu(con, sin).pvalue, 5)})
pl.DataFrame(filas)

Periodo,Estados con gobernador,Cambio promedio con gobernador (pp),Estados sin gobernador,Cambio promedio sin gobernador (pp),Valor p (prueba de Mann-Whitney)
str,i64,f64,i64,f64,f64
"""2012 → 2015""",9,-4.3,23,-17.6,0.00003
"""2018 → 2021""",15,-7.3,17,-11.7,0.00655


**Qué vemos:**

- Entre 2012 y 2015, la participación cayó **4.3 pp en los estados que eligieron gobernador y 17.6 pp en los demás**.
- Entre 2018 y 2021, 7.3 pp contra 11.7 pp.
- En los dos casos el valor p es muy pequeño: si elegir gobernador no influyera, sería muy raro ver una diferencia así entre los dos grupos de estados.

**Qué significa para el modelo:** la elección de gobernador el mismo día es probablemente **la variable más importante para el nivel de participación de cada estado** en una intermedia. Va en el modelo para 2027.

## 8 · Otras pistas para el modelo (por verificar)

Estas pistas salieron de la investigación, pero **todavía no las verificamos**. Se revisan en la exploración (paso 3):

- **Elección municipal el mismo día en municipios pequeños:** explica muchos de los saltos de participación alta (por ejemplo, Veracruz en 2021).
- **Crecimiento de la lista:** donde la lista crece rápido (periferias urbanas), se vota menos.
- **Credencialización de los jóvenes:** los de 18 años con credencial son pocos y parecen los más motivados. Su participación alta no significa que "los jóvenes ya votan".

## 9 · Decisiones

**Decididas por Sofía Ordaz el 19 de octubre de 2026.** Este análisis cambia una decisión del paso 1 y agrega tres nuevas. Quedan registradas en `docs/bitacora.md`.

| # | Decisión | Qué se decidió | Por qué |
|---|---|---|---|
| **D3 (ajustada)** | Qué hacer cuando hay mucho "no sabemos" | Distinguir el origen con las actas. **A (no hubo casilla):** calcular la participación con las actas sobre toda la lista y marcar "no participación comunitaria o por conflicto". **B (paquete perdido):** participación de las actas y marca de conflicto. **C (no capturado):** participación de las actas; si no hay actas, "sin dato" | Con la regla anterior, Cherán, Chicomuselo y Pantelhó, los de más abstención real, habrían quedado en gris |
| **D9 (nueva)** | Usar las actas oficiales como segunda fuente | Incorporar las actas por casilla de 2018, 2021 y 2024 | Permiten separar los tres orígenes y validar los datos |
| **D10 (nueva)** | Secciones donde nuestros datos y el acta no coinciden | Si nuestros datos marcan más de 10 % (y al menos 50) más "votó" que votos el acta, la sección queda "sin dato" | Dos fuentes oficiales no coinciden; el dato no es confiable |
| **D11 (nueva)** | Crecimiento atípico de la lista | No mostrarlo como "sospecha" en el mapa; usarlo como variable continua en el modelo | Casi todo es crecimiento urbano; solo un caso combina crecimiento atípico con salto de participación |

## Cómo comunicar estos hallazgos en el reporte

- **Decir** "participación atípica" o "dos fuentes oficiales no coinciden". **Nunca** "fraude", "voto de muertos" ni "turismo electoral".
- **Separar en cada afirmación** lo que dice el dato, lo que dice la prensa (atribuido) y lo que no se encontró.
- **Poner siempre la explicación legítima junto a la sospecha.**
- **No nombrar personas** ni vincular a autoridades con grupos criminales.

## Pendientes

- Conseguir las actas por casilla de 2018 y 2021 (los reportes de investigación ya las descargaron) para clasificar los orígenes en esos años.
- Lista oficial de municipios de Oaxaca que eligen por usos y costumbres: el sitio del IEEPCO no permite descargarla (ver `docs/investigacion/H-busquedas-pendientes.md`).
- Verificar las pistas de la sección 7.